In [1]:
# Core libraries
import ast
import re
import numpy as np
import pandas as pd

# Sklearn
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

# Model
from catboost import CatBoostRegressor

In [2]:
# Safely convert string → Python object
def safe_literal_eval(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) else []
    except Exception:
        return []

# Convert tags list → clean text
def parse_tags_to_text(x):
    vals = safe_literal_eval(x)
    if not isinstance(vals, list):
        return ""
    return " ".join(
        str(t).strip().lower().replace(" ", "_") for t in vals
    )

# Count items inside list-like columns
def count_list_items(x):
    v = safe_literal_eval(x)
    return len(v) if isinstance(v, list) else 0

# Total votes from ratings
def ratings_total_votes(x):
    v = safe_literal_eval(x)
    if not isinstance(v, list):
        return 0
    return sum(int(item.get("count", 0)) for item in v if isinstance(item, dict))

# Count specific rating type
def rating_count_by_name(x, wanted):
    v = safe_literal_eval(x)
    if not isinstance(v, list):
        return 0
    return sum(
        int(item.get("count", 0))
        for item in v
        if isinstance(item, dict) and str(item.get("name", "")).lower() == wanted
    )

# Entropy helper
def entropy_from_counts(counts):
    arr = np.array(counts, dtype=float)
    s = arr.sum()
    if s <= 0:
        return 0.0
    p = arr / s
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

# Ratings entropy
def ratings_entropy(x):
    v = safe_literal_eval(x)
    if not isinstance(v, list):
        return 0.0
    counts = [float(item.get("count", 0)) for item in v if isinstance(item, dict)]
    return entropy_from_counts(counts) if counts else 0.0

# Extract time-based features
def add_time_features(df, col):
    dt = pd.to_datetime(df[col], unit="s", errors="coerce")
    df[f"{col}_year"] = dt.dt.year
    df[f"{col}_month"] = dt.dt.month
    df[f"{col}_dow"] = dt.dt.dayofweek
    return df

# Basic text cleaning
def clean_text(s):
    return re.sub(r"\s+", " ", str(s).lower()).strip()

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "views"
ID_COL = "id"

# Target transformation
y = train[TARGET].clip(lower=0)
y_log = np.log1p(y)

test_ids = test[ID_COL].copy()

# Combine datasets for consistent preprocessing
full = pd.concat(
    [train.drop(columns=[TARGET]), test],
    axis=0,
    ignore_index=True
)

n_train = len(train)

In [4]:
# Fill missing values
full["speaker_occupation"] = full["speaker_occupation"].fillna("unknown")

# Time features
full = add_time_features(full, "film_date")
full = add_time_features(full, "published_date")
full["days_to_publish"] = (full["published_date"] - full["film_date"]) / 86400.0

# Numeric interactions
full["duration_per_speaker"] = full["duration"] / np.maximum(full["num_speaker"], 1)

# Text length features
full["title_len_chars"] = full["title"].astype(str).str.len()
full["title_len_words"] = full["title"].astype(str).str.split().str.len()
full["desc_len_chars"] = full["description"].astype(str).str.len()
full["desc_len_words"] = full["description"].astype(str).str.split().str.len()
full["name_len_words"] = full["name"].astype(str).str.split().str.len()

# List-based features
full["tags_count"] = full["tags"].apply(count_list_items)
full["related_talks_count"] = full["related_talks"].apply(count_list_items)
full["ratings_count_types"] = full["ratings"].apply(count_list_items)
full["ratings_total_votes"] = full["ratings"].apply(ratings_total_votes)
full["ratings_entropy"] = full["ratings"].apply(ratings_entropy)

In [5]:
rating_names = [
    "inspiring", "funny", "ingenious", "fascinating", "persuasive",
    "jaw-dropping", "beautiful", "courageous", "informative",
    "confusing", "obnoxious", "ok", "unconvincing", "longwinded"
]

for rn in rating_names:
    count_col = f"rating_{rn}_count"
    
    full[count_col] = full["ratings"].apply(
        lambda x, rn=rn: rating_count_by_name(x, rn)
    )
    
    full[f"rating_{rn}_ratio"] = (
        full[count_col] / np.maximum(full["ratings_total_votes"], 1)
    )

In [6]:
for col in ["main_speaker", "event", "speaker_occupation"]:
    freq = full.loc[:n_train - 1, col].value_counts()
    full[f"{col}_freq"] = full[col].map(freq).fillna(0)

In [7]:
drop_cols_tab = [
    "Unnamed: 0", "id", "url",
    "description", "name", "ratings",
    "related_talks", "tags", "title"
]

X_tab = full.drop(columns=drop_cols_tab)

X_tab_train = X_tab.iloc[:n_train].copy()
X_tab_test = X_tab.iloc[n_train:].copy()

cat_cols = ["event", "main_speaker", "speaker_occupation"]
cat_idx = [
    X_tab_train.columns.get_loc(c)
    for c in cat_cols if c in X_tab_train.columns
]

In [8]:
full["tags_text"] = full["tags"].apply(parse_tags_to_text)

full["text_all"] = (
    full["title"].map(clean_text) + " " +
    full["description"].map(clean_text) + " " +
    full["tags_text"].map(clean_text) + " " +
    full["speaker_occupation"].astype(str).map(clean_text) + " " +
    full["event"].astype(str).map(clean_text) + " " +
    full["main_speaker"].astype(str).map(clean_text)
)

text_train = full.loc[:n_train - 1, "text_all"].values
text_test = full.loc[n_train:, "text_all"].values

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=80000,
    strip_accents="unicode",
    sublinear_tf=True
)

X_txt_train = tfidf.fit_transform(text_train)
X_txt_test = tfidf.transform(text_test)

In [9]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_cat_log = np.zeros(n_train)
oof_ridge_log = np.zeros(n_train)

test_cat_log_folds = []
test_ridge_log_folds = []

In [10]:
for fold, (tr_idx, va_idx) in enumerate(kf.split(X_tab_train), 1):

    # Split data
    Xtr_tab, Xva_tab = X_tab_train.iloc[tr_idx], X_tab_train.iloc[va_idx]
    ytr, yva = y_log.iloc[tr_idx], y_log.iloc[va_idx]

    # -----------------
    # CatBoost model
    # -----------------
    cat = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        depth=6,
        learning_rate=0.03,
        n_estimators=4000,
        l2_leaf_reg=8,
        random_seed=2026 + fold,
        verbose=0
    )

    cat.fit(
        Xtr_tab, ytr,
        cat_features=cat_idx,
        eval_set=(Xva_tab, yva),
        use_best_model=True,
        early_stopping_rounds=200
    )

    oof_cat_log[va_idx] = cat.predict(Xva_tab)
    test_cat_log_folds.append(cat.predict(X_tab_test))

    # -----------------
    # Ridge model (text)
    # -----------------
    Xtr_txt, Xva_txt = X_txt_train[tr_idx], X_txt_train[va_idx]

    ridge = Ridge(alpha=8.0, random_state=42)
    ridge.fit(Xtr_txt, ytr)

    oof_ridge_log[va_idx] = ridge.predict(Xva_txt)
    test_ridge_log_folds.append(ridge.predict(X_txt_test))

    # Metrics
    fold_cat_msle = mean_squared_log_error(
        np.expm1(yva), np.clip(np.expm1(oof_cat_log[va_idx]), 0, None)
    )

    fold_ridge_msle = mean_squared_log_error(
        np.expm1(yva), np.clip(np.expm1(oof_ridge_log[va_idx]), 0, None)
    )

    print(f"Fold {fold} | Cat: {fold_cat_msle:.6f} | Ridge: {fold_ridge_msle:.6f}")

Fold 1 | Cat: 0.522040 | Ridge: 1.092719
Fold 2 | Cat: 0.120971 | Ridge: 0.587079
Fold 3 | Cat: 0.135948 | Ridge: 0.628015
Fold 4 | Cat: 0.141254 | Ridge: 0.645002
Fold 5 | Cat: 0.151536 | Ridge: 0.484396


In [11]:
oof_cat = np.clip(np.expm1(oof_cat_log), 0, None)
oof_ridge = np.clip(np.expm1(oof_ridge_log), 0, None)

msle_cat = mean_squared_log_error(y, oof_cat)
msle_ridge = mean_squared_log_error(y, oof_ridge)

print("CatBoost MSLE:", msle_cat)
print("Ridge MSLE:", msle_ridge)

CatBoost MSLE: 0.21434987565028127
Ridge MSLE: 0.687442317583043


In [12]:
best_w = 0
best_score = float("inf")

for w in np.linspace(0, 1, 101):
    blend = w * oof_cat + (1 - w) * oof_ridge
    score = mean_squared_log_error(y, np.clip(blend, 0, None))

    if score < best_score:
        best_score = score
        best_w = w

print("Best weight (CatBoost):", best_w)
print("Best blended MSLE:", best_score)

Best weight (CatBoost): 1.0
Best blended MSLE: 0.21434987565028127


In [13]:
test_cat = np.clip(np.expm1(np.mean(test_cat_log_folds, axis=0)), 0, None)
test_ridge = np.clip(np.expm1(np.mean(test_ridge_log_folds, axis=0)), 0, None)

test_blend = best_w * test_cat + (1 - best_w) * test_ridge
test_blend = np.clip(test_blend, 0, None)

# Save files
pd.DataFrame({ID_COL: test_ids, TARGET: test_blend}) \
    .to_csv("submission_blend.csv", index=False)

pd.DataFrame({ID_COL: test_ids, TARGET: test_cat}) \
    .to_csv("submission_catboost.csv", index=False)

pd.DataFrame({ID_COL: test_ids, TARGET: test_ridge}) \
    .to_csv("submission_ridge_tfidf.csv", index=False)

print("Submissions saved!")

Submissions saved!


import ast
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

from catboost import CatBoostRegressor


# -----------------------------
# Helpers
# -----------------------------
def safe_literal_eval(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) else []
    except Exception:
        return []

def parse_tags_to_text(x):
    vals = safe_literal_eval(x)
    if not isinstance(vals, list):
        return ""
    cleaned = []
    for t in vals:
        t = str(t).strip().lower().replace(" ", "_")
        cleaned.append(t)
    return " ".join(cleaned)

def count_list_items(x):
    v = safe_literal_eval(x)
    return len(v) if isinstance(v, list) else 0

def ratings_total_votes(x):
    v = safe_literal_eval(x)
    if not isinstance(v, list):
        return 0
    s = 0
    for item in v:
        if isinstance(item, dict):
            s += int(item.get("count", 0))
    return s

def rating_count_by_name(x, wanted):
    v = safe_literal_eval(x)
    if not isinstance(v, list):
        return 0
    total = 0
    for item in v:
        if isinstance(item, dict) and str(item.get("name", "")).lower() == wanted:
            total += int(item.get("count", 0))
    return total

def entropy_from_counts(counts):
    arr = np.array(counts, dtype=float)
    s = arr.sum()
    if s <= 0:
        return 0.0
    p = arr / s
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

def ratings_entropy(x):
    v = safe_literal_eval(x)
    if not isinstance(v, list):
        return 0.0
    counts = []
    for item in v:
        if isinstance(item, dict):
            counts.append(float(item.get("count", 0)))
    if not counts:
        return 0.0
    return entropy_from_counts(counts)

def add_time_features(df, col):
    dt = pd.to_datetime(df[col], unit="s", errors="coerce")
    df[f"{col}_year"] = dt.dt.year
    df[f"{col}_month"] = dt.dt.month
    df[f"{col}_dow"] = dt.dt.dayofweek
    return df

def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s


# -----------------------------
# Load data
# -----------------------------
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "views"
ID_COL = "id"

y = train[TARGET].clip(lower=0)
y_log = np.log1p(y)

test_ids = test[ID_COL].copy()

# Combine for consistent transformations
full = pd.concat([train.drop(columns=[TARGET]), test], axis=0, ignore_index=True)
n_train = len(train)


# -----------------------------
# Feature engineering (tabular)
# -----------------------------
full["speaker_occupation"] = full["speaker_occupation"].fillna("unknown")

# time
full = add_time_features(full, "film_date")
full = add_time_features(full, "published_date")
full["days_to_publish"] = (full["published_date"] - full["film_date"]) / 86400.0

# numeric interactions
full["duration_per_speaker"] = full["duration"] / np.maximum(full["num_speaker"], 1)

# text lengths
full["title_len_chars"] = full["title"].astype(str).str.len()
full["title_len_words"] = full["title"].astype(str).str.split().str.len()
full["desc_len_chars"] = full["description"].astype(str).str.len()
full["desc_len_words"] = full["description"].astype(str).str.split().str.len()
full["name_len_words"] = full["name"].astype(str).str.split().str.len()

# parsed list features
full["tags_count"] = full["tags"].apply(count_list_items)
full["related_talks_count"] = full["related_talks"].apply(count_list_items)
full["ratings_count_types"] = full["ratings"].apply(count_list_items)
full["ratings_total_votes"] = full["ratings"].apply(ratings_total_votes)
full["ratings_entropy"] = full["ratings"].apply(ratings_entropy)

# rating-specific counts/ratios (common TED rating names)
rating_names = [
    "inspiring", "funny", "ingenious", "fascinating", "persuasive",
    "jaw-dropping", "beautiful", "courageous", "informative",
    "confusing", "obnoxious", "ok", "unconvincing", "longwinded"
]
for rn in rating_names:
    colc = f"rating_{rn}_count"
    full[colc] = full["ratings"].apply(lambda x, rn=rn: rating_count_by_name(x, rn))
    full[f"rating_{rn}_ratio"] = full[colc] / np.maximum(full["ratings_total_votes"], 1)

# freq encoding from train only
for col in ["main_speaker", "event", "speaker_occupation"]:
    freq = full.loc[:n_train-1, col].value_counts()
    full[f"{col}_freq"] = full[col].map(freq).fillna(0)

# Prepare tabular matrix for CatBoost
drop_cols_tab = [
    "Unnamed: 0", "id", "url",
    "description", "name", "ratings", "related_talks", "tags", "title"
]
X_tab = full.drop(columns=drop_cols_tab).copy()
X_tab_train = X_tab.iloc[:n_train].copy()
X_tab_test = X_tab.iloc[n_train:].copy()

cat_cols = ["event", "main_speaker", "speaker_occupation"]
cat_idx = [X_tab_train.columns.get_loc(c) for c in cat_cols if c in X_tab_train.columns]


# -----------------------------
# Text matrix for Ridge
# -----------------------------
full["tags_text"] = full["tags"].apply(parse_tags_to_text)

full["text_all"] = (
    full["title"].map(clean_text) + " " +
    full["description"].map(clean_text) + " " +
    full["tags_text"].map(clean_text) + " " +
    full["speaker_occupation"].astype(str).map(clean_text) + " " +
    full["event"].astype(str).map(clean_text) + " " +
    full["main_speaker"].astype(str).map(clean_text)
)

text_train = full.loc[:n_train-1, "text_all"].values
text_test = full.loc[n_train:, "text_all"].values

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=80000,
    strip_accents="unicode",
    sublinear_tf=True
)
X_txt_train = tfidf.fit_transform(text_train)
X_txt_test = tfidf.transform(text_test)


# -----------------------------
# CV train both models
# -----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_cat_log = np.zeros(n_train)
oof_ridge_log = np.zeros(n_train)

test_cat_log_folds = []
test_ridge_log_folds = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_tab_train), 1):
    Xtr_tab, Xva_tab = X_tab_train.iloc[tr_idx], X_tab_train.iloc[va_idx]
    ytr, yva = y_log.iloc[tr_idx], y_log.iloc[va_idx]

    # CatBoost
    cat = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        depth=6,
        learning_rate=0.03,
        n_estimators=4000,
        l2_leaf_reg=8,
        random_seed=2026 + fold,
        verbose=0
    )
    cat.fit(
        Xtr_tab, ytr,
        cat_features=cat_idx,
        eval_set=(Xva_tab, yva),
        use_best_model=True,
        early_stopping_rounds=200
    )
    oof_cat_log[va_idx] = cat.predict(Xva_tab)
    test_cat_log_folds.append(cat.predict(X_tab_test))

    # Ridge TF-IDF
    Xtr_txt, Xva_txt = X_txt_train[tr_idx], X_txt_train[va_idx]
    ridge = Ridge(alpha=8.0, random_state=42)
    ridge.fit(Xtr_txt, ytr)
    oof_ridge_log[va_idx] = ridge.predict(Xva_txt)
    test_ridge_log_folds.append(ridge.predict(X_txt_test))

    # Fold metrics quick print
    fold_cat_msle = mean_squared_log_error(
        np.expm1(yva), np.clip(np.expm1(oof_cat_log[va_idx]), 0, None)
    )
    fold_ridge_msle = mean_squared_log_error(
        np.expm1(yva), np.clip(np.expm1(oof_ridge_log[va_idx]), 0, None)
    )
    print(f"Fold {fold} | Cat MSLE: {fold_cat_msle:.6f} | Ridge MSLE: {fold_ridge_msle:.6f}")

# Overall OOF metrics
oof_cat = np.clip(np.expm1(oof_cat_log), 0, None)
oof_ridge = np.clip(np.expm1(oof_ridge_log), 0, None)

msle_cat = mean_squared_log_error(y, oof_cat)
msle_ridge = mean_squared_log_error(y, oof_ridge)

print(f"\nOOF CatBoost MSLE: {msle_cat:.6f}")
print(f"OOF Ridge TFIDF MSLE: {msle_ridge:.6f}")

# -----------------------------
# Find best blend weight on OOF
# -----------------------------
best_w = None
best_msle = 1e18
for w in np.linspace(0, 1, 101):
    blend = w * oof_cat + (1 - w) * oof_ridge
    s = mean_squared_log_error(y, np.clip(blend, 0, None))
    if s < best_msle:
        best_msle = s
        best_w = w

print(f"Best blend weight (CatBoost): {best_w:.2f}")
print(f"Best OOF Blend MSLE: {best_msle:.6f}")

# -----------------------------
# Predict test + save
# -----------------------------
test_cat = np.clip(np.expm1(np.mean(test_cat_log_folds, axis=0)), 0, None)
test_ridge = np.clip(np.expm1(np.mean(test_ridge_log_folds, axis=0)), 0, None)

test_blend = best_w * test_cat + (1 - best_w) * test_ridge
test_blend = np.clip(test_blend, 0, None)

sub_blend = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: test_blend
})
sub_blend.to_csv("submission_blend.csv", index=False)
print("Saved: submission_blend.csv")

# also save single-model submissions for safety
pd.DataFrame({ID_COL: test_ids, TARGET: test_cat}).to_csv("submission_catboost.csv", index=False)
pd.DataFrame({ID_COL: test_ids, TARGET: test_ridge}).to_csv("submission_ridge_tfidf.csv", index=False)
print("Saved: submission_catboost.csv, submission_ridge_tfidf.csv")

In [14]:
print("train shape:", train.shape)
print("test shape:", test.shape)
print("\nColumns:\n", train.columns.tolist())
print("\nDtypes:\n", train.dtypes)

print("\nTrain head:\n", train.head())
print("\nTest head:\n", test.head())

print("\nMissing ratio:\n", (train.isnull().mean().sort_values(ascending=False)*100).round(2))

cat_cols = train.select_dtypes(include=["object", "category"]).columns
print("\nCategorical cardinality:\n", train[cat_cols].nunique().sort_values(ascending=False))


train shape: (2040, 18)
test shape: (510, 17)

Columns:
 ['Unnamed: 0', 'description', 'duration', 'event', 'film_date', 'languages', 'main_speaker', 'name', 'num_speaker', 'published_date', 'ratings', 'related_talks', 'speaker_occupation', 'tags', 'title', 'url', 'views', 'id']

Dtypes:
 Unnamed: 0             int64
description           object
duration               int64
event                 object
film_date              int64
languages              int64
main_speaker          object
name                  object
num_speaker            int64
published_date         int64
ratings               object
related_talks         object
speaker_occupation    object
tags                  object
title                 object
url                   object
views                  int64
id                     int64
dtype: object

Train head:
    Unnamed: 0                                        description  duration  \
0        1601  How do we solve the problem of the suburbs? Ur...      1016   
1   

In [1]:
# =============================================================================
# TED Talk View Prediction — v4 (Leak-Free CV + Consistent Log Blend)
#
# Improvements vs v3:
#   1) Target encoding is now truly leak-free with OUTER-FOLD-local encoding
#   2) Blend optimization and test-time blending are both done in LOG SPACE
#   3) talk_age_days uses TRAIN-ONLY reference date
#   4) Added extra transcript/style features
#   5) Added finer blend search
#
# Models:
#   • CatBoostRegressor
#   • LightGBMRegressor
#   • Ridge on short text
#   • Ridge on transcript text
#
# Metric:
#   • Mean Squared Log Error (MSLE)
# =============================================================================

# pip install catboost lightgbm scikit-learn pandas numpy scipy

import ast
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from scipy.sparse import hstack, csr_matrix

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import lightgbm as lgb_module

warnings.filterwarnings("ignore")

# =============================================================================
# Config
# =============================================================================

TARGET = "views"
ID_COL = "id"
N_SPLITS = 5
SEED = 42

# =============================================================================
# Helpers
# =============================================================================

def safe_eval(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) else []
    except Exception:
        return []

def clean_text(s):
    return re.sub(r"\s+", " ", str(s).lower()).strip()

def parse_tags_to_text(x):
    vals = safe_eval(x)
    if not isinstance(vals, list):
        return ""
    return " ".join(str(t).strip().lower().replace(" ", "_") for t in vals)

def count_list_items(x):
    v = safe_eval(x)
    return len(v) if isinstance(v, list) else 0

def ratings_total_votes(x):
    v = safe_eval(x)
    if not isinstance(v, list):
        return 0
    return sum(int(item.get("count", 0)) for item in v if isinstance(item, dict))

def rating_count_by_name(x, wanted):
    v = safe_eval(x)
    if not isinstance(v, list):
        return 0
    wanted = str(wanted).lower()
    return sum(
        int(item.get("count", 0))
        for item in v
        if isinstance(item, dict) and str(item.get("name", "")).lower() == wanted
    )

def entropy_from_counts(counts):
    arr = np.array(counts, dtype=float)
    s = arr.sum()
    if s <= 0:
        return 0.0
    p = arr / s
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

def ratings_entropy(x):
    v = safe_eval(x)
    if not isinstance(v, list):
        return 0.0
    counts = [float(item.get("count", 0)) for item in v if isinstance(item, dict)]
    return entropy_from_counts(counts) if counts else 0.0

def add_time_features(df, col):
    dt = pd.to_datetime(df[col], unit="s", errors="coerce")
    df[f"{col}_year"] = dt.dt.year
    df[f"{col}_month"] = dt.dt.month
    df[f"{col}_dow"] = dt.dt.dayofweek
    return df

def related_talks_view_stats(x):
    v = safe_eval(x)
    if not isinstance(v, list) or len(v) == 0:
        return 0.0, 0.0, 0, 0.0, 0.0, 0.0
    counts = [item.get("viewed_count", 0) for item in v if isinstance(item, dict)]
    counts = [float(c) for c in counts if c and c > 0]
    if not counts:
        return 0.0, 0.0, 0, 0.0, 0.0, 0.0
    return (
        float(np.mean(counts)),
        float(np.max(counts)),
        len(counts),
        float(np.median(counts)),
        float(np.std(counts)),
        float(np.min(counts)),
    )

def safe_word_stats(text):
    text = str(text)
    words = re.findall(r"\b\w+\b", text.lower())
    n = len(words)
    if n == 0:
        return 0.0, 0.0, 0.0, 0.0
    uniq = len(set(words))
    avg_len = float(np.mean([len(w) for w in words]))
    long_ratio = float(np.mean([len(w) >= 7 for w in words]))
    sent_count = max(len(re.findall(r"[.!?]+", text)), 1)
    avg_sent_len = n / sent_count
    return uniq / n, avg_len, long_ratio, avg_sent_len

def oof_target_encode_train_valid_test(
    tr_series,
    ytr_log,
    va_series,
    te_series,
    n_splits=5,
    smooth=20,
    random_state=42
):
    """
    Leak-free target encoding inside an OUTER fold.

    tr_series: outer-train categorical series
    ytr_log:   outer-train target in log1p space
    va_series: outer-valid categorical series
    te_series: test categorical series

    Returns:
        tr_enc  : inner-OOF encoding for outer-train rows
        va_enc  : encoding for outer-valid using full outer-train stats
        te_enc  : encoding for test using full outer-train stats
    """
    tr_series = tr_series.reset_index(drop=True)
    ytr_log = ytr_log.reset_index(drop=True)
    va_series = va_series.reset_index(drop=True)
    te_series = te_series.reset_index(drop=True)

    global_mean = ytr_log.mean()
    tr_enc = np.full(len(tr_series), global_mean, dtype=float)

    inner_kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    # Inner OOF for outer-train
    for inner_tr_idx, inner_va_idx in inner_kf.split(tr_series):
        tmp = pd.DataFrame({
            "cat": tr_series.iloc[inner_tr_idx].values,
            "y": ytr_log.iloc[inner_tr_idx].values
        })
        stats = tmp.groupby("cat")["y"].agg(["mean", "count"])
        stats["enc"] = (
            stats["count"] * stats["mean"] + smooth * global_mean
        ) / (stats["count"] + smooth)

        tr_enc[inner_va_idx] = (
            tr_series.iloc[inner_va_idx]
            .map(stats["enc"])
            .fillna(global_mean)
            .values
        )

    # Full outer-train stats for valid/test
    full_stats = pd.DataFrame({
        "cat": tr_series.values,
        "y": ytr_log.values
    }).groupby("cat")["y"].agg(["mean", "count"])
    full_stats["enc"] = (
        full_stats["count"] * full_stats["mean"] + smooth * global_mean
    ) / (full_stats["count"] + smooth)

    va_enc = va_series.map(full_stats["enc"]).fillna(global_mean).values
    te_enc = te_series.map(full_stats["enc"]).fillna(global_mean).values

    return tr_enc, va_enc, te_enc

def msle_from_log(y_true, pred_log):
    pred = np.clip(np.expm1(pred_log), 0, None)
    return mean_squared_log_error(y_true, pred)

def refine_blend_weights(oof_logs, y_true):
    """
    Two-stage search:
      1) coarse grid step=0.05
      2) local refinement around best with step=0.01
    """
    best_score = float("inf")
    best_w = (1.0, 0.0, 0.0, 0.0)

    # Stage 1: coarse
    coarse = np.arange(0, 1.0001, 0.05)
    for w1 in coarse:
        for w2 in coarse:
            if w1 + w2 > 1:
                continue
            for w3 in coarse:
                if w1 + w2 + w3 > 1:
                    continue
                w4 = 1.0 - w1 - w2 - w3
                blend_log = (
                    w1 * oof_logs[0] +
                    w2 * oof_logs[1] +
                    w3 * oof_logs[2] +
                    w4 * oof_logs[3]
                )
                score = msle_from_log(y_true, blend_log)
                if score < best_score:
                    best_score = score
                    best_w = (w1, w2, w3, w4)

    # Stage 2: fine refinement around best
    w1c, w2c, w3c, _ = best_w
    ranges1 = np.arange(max(0, w1c - 0.08), min(1, w1c + 0.08) + 1e-9, 0.01)
    ranges2 = np.arange(max(0, w2c - 0.08), min(1, w2c + 0.08) + 1e-9, 0.01)
    ranges3 = np.arange(max(0, w3c - 0.08), min(1, w3c + 0.08) + 1e-9, 0.01)

    for w1 in ranges1:
        for w2 in ranges2:
            if w1 + w2 > 1:
                continue
            for w3 in ranges3:
                if w1 + w2 + w3 > 1:
                    continue
                w4 = 1.0 - w1 - w2 - w3
                blend_log = (
                    w1 * oof_logs[0] +
                    w2 * oof_logs[1] +
                    w3 * oof_logs[2] +
                    w4 * oof_logs[3]
                )
                score = msle_from_log(y_true, blend_log)
                if score < best_score:
                    best_score = score
                    best_w = (round(w1, 4), round(w2, 4), round(w3, 4), round(w4, 4))

    return best_score, best_w

# =============================================================================
# Load data
# =============================================================================

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
trans = pd.read_csv("transcripts.csv").rename(columns={"transcript": "transcript_text"})

y = train[TARGET].clip(lower=0)
y_log = np.log1p(y)
test_ids = test[ID_COL].copy()

# =============================================================================
# Combine for preprocessing
# =============================================================================

full = pd.concat(
    [train.drop(columns=[TARGET]), test],
    axis=0,
    ignore_index=True
)
n_train = len(train)

# =============================================================================
# Join transcripts (deduplicate first)
# =============================================================================

trans["transcript_text"] = trans["transcript_text"].fillna("").astype(str)
trans = trans.sort_values("transcript_text", key=lambda s: s.str.len(), ascending=False)
trans = trans.drop_duplicates(subset="url", keep="first")

full = full.merge(trans[["url", "transcript_text"]], on="url", how="left")
full["transcript_text"] = full["transcript_text"].fillna("")
full["transcript_missing"] = full["transcript_text"].eq("").astype(int)

# =============================================================================
# Basic fixes
# =============================================================================

for col in ["speaker_occupation", "event", "main_speaker", "description", "name", "title"]:
    if col in full.columns:
        full[col] = full[col].fillna("unknown").astype(str)

for col in ["ratings", "related_talks", "tags"]:
    if col in full.columns:
        full[col] = full[col].fillna("[]")

full["languages"] = full["languages"].fillna(0)

# =============================================================================
# Time features
# =============================================================================

full = add_time_features(full, "film_date")
full = add_time_features(full, "published_date")
full["days_to_publish"] = (full["published_date"] - full["film_date"]) / 86400.0

# TRAIN-ONLY reference date to avoid test-informed age feature
max_pub_train = train["published_date"].max()
full["talk_age_days"] = (max_pub_train - full["published_date"]) / 86400.0

# =============================================================================
# Numeric / interactions
# =============================================================================

full["languages_sq"] = full["languages"] ** 2
full["duration_per_speaker"] = full["duration"] / np.maximum(full["num_speaker"], 1)

full["duration_bucket"] = pd.cut(
    full["duration"],
    bins=[0, 300, 600, 900, 1200, 99999],
    labels=False,
    include_lowest=True
).astype(float)

# =============================================================================
# Basic text length features
# =============================================================================

full["title_len_chars"] = full["title"].str.len()
full["title_len_words"] = full["title"].str.split().str.len()
full["desc_len_chars"] = full["description"].str.len()
full["desc_len_words"] = full["description"].str.split().str.len()
full["name_len_words"] = full["name"].str.split().str.len()

full["transcript_len_chars"] = full["transcript_text"].str.len()
full["transcript_len_words"] = full["transcript_text"].str.split().str.len()

# =============================================================================
# Transcript engagement / style features
# =============================================================================

low_trans = full["transcript_text"].str.lower()

full["laughter_count"] = low_trans.str.count(r"\(laughter\)")
full["applause_count"] = low_trans.str.count(r"\(applause\)")
full["question_count"] = full["transcript_text"].str.count(r"\?")
full["exclamation_count"] = full["transcript_text"].str.count(r"!")
full["digit_count"] = full["transcript_text"].str.count(r"\d")
full["parenthetical_count"] = full["transcript_text"].str.count(r"\(")

full["laughter_per_word"] = full["laughter_count"] / np.maximum(full["transcript_len_words"], 1)
full["applause_per_word"] = full["applause_count"] / np.maximum(full["transcript_len_words"], 1)
full["question_per_word"] = full["question_count"] / np.maximum(full["transcript_len_words"], 1)
full["exclamation_per_word"] = full["exclamation_count"] / np.maximum(full["transcript_len_words"], 1)
full["digit_ratio"] = full["digit_count"] / np.maximum(full["transcript_len_chars"], 1)
full["parenthetical_ratio"] = full["parenthetical_count"] / np.maximum(full["transcript_len_words"], 1)

word_stats = full["transcript_text"].apply(safe_word_stats)
full["trans_lex_div"] = word_stats.apply(lambda x: x[0])
full["trans_avg_word_len"] = word_stats.apply(lambda x: x[1])
full["trans_long_word_ratio"] = word_stats.apply(lambda x: x[2])
full["trans_avg_sent_len"] = word_stats.apply(lambda x: x[3])

# =============================================================================
# Related talks features
# =============================================================================

rt_stats = full["related_talks"].apply(related_talks_view_stats)
full["rt_mean_views"] = rt_stats.apply(lambda x: x[0])
full["rt_max_views"] = rt_stats.apply(lambda x: x[1])
full["rt_count"] = rt_stats.apply(lambda x: x[2])
full["rt_median_views"] = rt_stats.apply(lambda x: x[3])
full["rt_std_views"] = rt_stats.apply(lambda x: x[4])
full["rt_min_views"] = rt_stats.apply(lambda x: x[5])

full["log_rt_mean_views"] = np.log1p(full["rt_mean_views"])
full["log_rt_max_views"] = np.log1p(full["rt_max_views"])
full["log_rt_median_views"] = np.log1p(full["rt_median_views"])
full["log_rt_min_views"] = np.log1p(full["rt_min_views"])

# =============================================================================
# Ratings features
# =============================================================================

full["tags_count"] = full["tags"].apply(count_list_items)
full["related_talks_count"] = full["related_talks"].apply(count_list_items)
full["ratings_count_types"] = full["ratings"].apply(count_list_items)
full["ratings_total_votes"] = full["ratings"].apply(ratings_total_votes)
full["ratings_entropy"] = full["ratings"].apply(ratings_entropy)

rating_names = [
    "inspiring", "funny", "ingenious", "fascinating", "persuasive",
    "jaw-dropping", "beautiful", "courageous", "informative",
    "confusing", "obnoxious", "ok", "unconvincing", "longwinded"
]

for rn in rating_names:
    count_col = f"rating_{rn}_count"
    full[count_col] = full["ratings"].apply(lambda x, rn=rn: rating_count_by_name(x, rn))
    full[f"rating_{rn}_ratio"] = full[count_col] / np.maximum(full["ratings_total_votes"], 1)

positive_names = [
    "inspiring", "fascinating", "ingenious", "jaw-dropping",
    "beautiful", "courageous", "informative", "funny", "persuasive"
]
negative_names = ["confusing", "obnoxious", "unconvincing", "longwinded"]

full["pos_votes"] = full[[f"rating_{r}_count" for r in positive_names]].sum(axis=1)
full["neg_votes"] = full[[f"rating_{r}_count" for r in negative_names]].sum(axis=1)
full["pos_ratio"] = full["pos_votes"] / np.maximum(full["ratings_total_votes"], 1)
full["neg_ratio"] = full["neg_votes"] / np.maximum(full["ratings_total_votes"], 1)
full["pos_neg_ratio"] = full["pos_votes"] / np.maximum(full["neg_votes"], 1)
full["low_votes_flag"] = (full["ratings_total_votes"] < 50).astype(int)

# =============================================================================
# Frequency features from FULL TRAIN ONLY (allowed; no target used)
# =============================================================================

train_only = full.iloc[:n_train].copy()

for col in ["main_speaker", "event", "speaker_occupation"]:
    freq = train_only[col].value_counts()
    full[f"{col}_freq"] = full[col].map(freq).fillna(0)

speaker_counts = train_only["main_speaker"].value_counts()
event_counts = train_only["event"].value_counts()

full["is_repeat_speaker"] = full["main_speaker"].map(speaker_counts).gt(1).astype(int)
full["speaker_event_key"] = full["main_speaker"].astype(str) + " || " + full["event"].astype(str)
speaker_event_freq = (train_only["main_speaker"].astype(str) + " || " + train_only["event"].astype(str)).value_counts()
full["speaker_event_freq"] = full["speaker_event_key"].map(speaker_event_freq).fillna(0)

# =============================================================================
# Text fields
# =============================================================================

full["tags_text"] = full["tags"].apply(parse_tags_to_text)

full["text_short"] = (
    full["title"].map(clean_text) + " " +
    full["description"].map(clean_text) + " " +
    full["tags_text"].map(clean_text) + " " +
    full["speaker_occupation"].astype(str).map(clean_text) + " " +
    full["event"].astype(str).map(clean_text) + " " +
    full["main_speaker"].astype(str).map(clean_text)
)
full["text_transcript"] = full["transcript_text"].map(clean_text)

# =============================================================================
# Base tabular matrices (NO target encoding yet)
# =============================================================================

drop_cols_tab = [
    "Unnamed: 0", "id", "url",
    "description", "name", "ratings",
    "related_talks", "tags", "title",
    "transcript_text", "tags_text", "text_short", "text_transcript",
    "speaker_event_key",
]

X_base = full.drop(columns=[c for c in drop_cols_tab if c in full.columns]).copy()
X_base_train = X_base.iloc[:n_train].copy()
X_base_test = X_base.iloc[n_train:].copy()

cat_cols = ["event", "main_speaker", "speaker_occupation"]
cat_cols = [c for c in cat_cols if c in X_base_train.columns]

# =============================================================================
# TF-IDF fit on TRAIN ONLY
# =============================================================================

short_word_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=60000,
    strip_accents="unicode",
    sublinear_tf=True
)

short_char_vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_features=30000,
    strip_accents="unicode",
    sublinear_tf=True
)

trans_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_features=100000,
    strip_accents="unicode",
    sublinear_tf=True
)

short_word_vec.fit(full.loc[:n_train - 1, "text_short"].values)
short_char_vec.fit(full.loc[:n_train - 1, "text_short"].values)
trans_vec.fit(full.loc[:n_train - 1, "text_transcript"].values)

X_short_word_train = short_word_vec.transform(full.loc[:n_train - 1, "text_short"].values)
X_short_word_test = short_word_vec.transform(full.loc[n_train:, "text_short"].values)

X_short_char_train = short_char_vec.transform(full.loc[:n_train - 1, "text_short"].values)
X_short_char_test = short_char_vec.transform(full.loc[n_train:, "text_short"].values)

X_short_train = hstack([X_short_word_train, X_short_char_train]).tocsr()
X_short_test = hstack([X_short_word_test, X_short_char_test]).tocsr()

X_trans_train = trans_vec.transform(full.loc[:n_train - 1, "text_transcript"].values)
X_trans_test = trans_vec.transform(full.loc[n_train:, "text_transcript"].values)

# =============================================================================
# Outer CV
# =============================================================================

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_cat_log = np.zeros(n_train)
oof_lgbm_log = np.zeros(n_train)
oof_ridge_log = np.zeros(n_train)
oof_rtrans_log = np.zeros(n_train)

test_cat_log_folds = []
test_lgbm_log_folds = []
test_ridge_log_folds = []
test_rtrans_log_folds = []

te_cols = ["main_speaker", "event", "speaker_occupation"]

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_base_train), 1):
    print(f"\n{'=' * 70}")
    print(f"Fold {fold}/{N_SPLITS}")
    print(f"{'=' * 70}")

    Xtr_tab = X_base_train.iloc[tr_idx].copy()
    Xva_tab = X_base_train.iloc[va_idx].copy()
    Xte_tab = X_base_test.copy()

    ytr = y_log.iloc[tr_idx].reset_index(drop=True)
    yva = y_log.iloc[va_idx].reset_index(drop=True)

    # -------------------------------------------------------------------------
    # Fold-local target encoding (TRUE leak-free)
    # -------------------------------------------------------------------------
    for col in te_cols:
        tr_enc, va_enc, te_enc = oof_target_encode_train_valid_test(
            tr_series=X_base_train.iloc[tr_idx][col],
            ytr_log=y_log.iloc[tr_idx],
            va_series=X_base_train.iloc[va_idx][col],
            te_series=X_base_test[col],
            n_splits=5,
            smooth=20,
            random_state=SEED + fold
        )
        Xtr_tab[f"{col}_target_enc"] = tr_enc
        Xva_tab[f"{col}_target_enc"] = va_enc
        Xte_tab[f"{col}_target_enc"] = te_enc

    # -------------------------------------------------------------------------
    # CatBoost
    # -------------------------------------------------------------------------
    cat_idx = [Xtr_tab.columns.get_loc(c) for c in cat_cols]

    cat_model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        depth=6,
        learning_rate=0.025,
        n_estimators=6000,
        l2_leaf_reg=5.0,
        min_data_in_leaf=10,
        bagging_temperature=0.5,
        random_strength=1.0,
        random_seed=2026 + fold,
        verbose=0,
    )

    cat_model.fit(
        Xtr_tab,
        ytr,
        cat_features=cat_idx,
        eval_set=(Xva_tab, yva),
        use_best_model=True,
        early_stopping_rounds=300,
    )

    cat_val_log = cat_model.predict(Xva_tab)
    cat_test_log = cat_model.predict(Xte_tab)

    oof_cat_log[va_idx] = cat_val_log
    test_cat_log_folds.append(cat_test_log)

    print(
        f"CatBoost        MSLE: "
        f"{mean_squared_log_error(np.expm1(yva), np.clip(np.expm1(cat_val_log), 0, None)):.6f} "
        f"(best_iter={cat_model.best_iteration_})"
    )

    # -------------------------------------------------------------------------
    # LightGBM
    # -------------------------------------------------------------------------
    Xtr_lgbm = Xtr_tab.copy()
    Xva_lgbm = Xva_tab.copy()
    Xte_lgbm = Xte_tab.copy()

    for c in cat_cols:
        Xtr_lgbm[c] = Xtr_lgbm[c].astype("category")
        Xva_lgbm[c] = Xva_lgbm[c].astype("category")
        Xte_lgbm[c] = Xte_lgbm[c].astype("category")

    lgbm_model = LGBMRegressor(
        n_estimators=6000,
        learning_rate=0.025,
        num_leaves=63,
        max_depth=-1,
        reg_alpha=0.5,
        reg_lambda=3.0,
        min_child_samples=30,
        colsample_bytree=0.8,
        subsample=0.8,
        subsample_freq=1,
        random_state=2026 + fold,
        n_jobs=-1,
        verbose=-1,
    )

    lgbm_model.fit(
        Xtr_lgbm,
        ytr,
        eval_set=[(Xva_lgbm, yva)],
        callbacks=[
            lgb_module.early_stopping(300, verbose=False),
            lgb_module.log_evaluation(-1),
        ],
        categorical_feature=cat_cols,
    )

    lgbm_val_log = lgbm_model.predict(Xva_lgbm)
    lgbm_test_log = lgbm_model.predict(Xte_lgbm)

    oof_lgbm_log[va_idx] = lgbm_val_log
    test_lgbm_log_folds.append(lgbm_test_log)

    print(
        f"LightGBM        MSLE: "
        f"{mean_squared_log_error(np.expm1(yva), np.clip(np.expm1(lgbm_val_log), 0, None)):.6f} "
        f"(best_iter={lgbm_model.best_iteration_})"
    )

    # -------------------------------------------------------------------------
    # Ridge — short text (word + char n-grams)
    # -------------------------------------------------------------------------
    Xtr_short = X_short_train[tr_idx]
    Xva_short = X_short_train[va_idx]

    ridge_meta = Ridge(alpha=8.0, random_state=SEED)
    ridge_meta.fit(Xtr_short, ytr)

    ridge_val_log = ridge_meta.predict(Xva_short)
    ridge_test_log = ridge_meta.predict(X_short_test)

    oof_ridge_log[va_idx] = ridge_val_log
    test_ridge_log_folds.append(ridge_test_log)

    print(
        f"Ridge(meta)     MSLE: "
        f"{mean_squared_log_error(np.expm1(yva), np.clip(np.expm1(ridge_val_log), 0, None)):.6f}"
    )

    # -------------------------------------------------------------------------
    # Ridge — transcript
    # -------------------------------------------------------------------------
    Xtr_tr = X_trans_train[tr_idx]
    Xva_tr = X_trans_train[va_idx]

    ridge_trans = Ridge(alpha=10.0, random_state=SEED)
    ridge_trans.fit(Xtr_tr, ytr)

    rtrans_val_log = ridge_trans.predict(Xva_tr)
    rtrans_test_log = ridge_trans.predict(X_trans_test)

    oof_rtrans_log[va_idx] = rtrans_val_log
    test_rtrans_log_folds.append(rtrans_test_log)

    print(
        f"Ridge(trans)    MSLE: "
        f"{mean_squared_log_error(np.expm1(yva), np.clip(np.expm1(rtrans_val_log), 0, None)):.6f}"
    )

# =============================================================================
# OOF Summary
# =============================================================================

print("\n" + "=" * 70)
print("OOF MSLE Summary")
print("=" * 70)
print(f"CatBoost:         {msle_from_log(y, oof_cat_log):.6f}")
print(f"LightGBM:         {msle_from_log(y, oof_lgbm_log):.6f}")
print(f"Ridge(meta):      {msle_from_log(y, oof_ridge_log):.6f}")
print(f"Ridge(trans):     {msle_from_log(y, oof_rtrans_log):.6f}")

# =============================================================================
# Blend optimization in LOG SPACE
# =============================================================================

print("\nSearching blend weights in log-space...")
best_score, best_weights = refine_blend_weights(
    oof_logs=[oof_cat_log, oof_lgbm_log, oof_ridge_log, oof_rtrans_log],
    y_true=y
)

w_cat, w_lgbm, w_ridge, w_rtrans = best_weights

print(f"\nBest OOF blend MSLE: {best_score:.6f}")
print(f"w_catboost   = {w_cat:.4f}")
print(f"w_lgbm       = {w_lgbm:.4f}")
print(f"w_ridge_meta = {w_ridge:.4f}")
print(f"w_ridge_tr   = {w_rtrans:.4f}")

# =============================================================================
# Aggregate fold predictions (still in LOG SPACE)
# =============================================================================

test_cat_log = np.mean(test_cat_log_folds, axis=0)
test_lgbm_log = np.mean(test_lgbm_log_folds, axis=0)
test_ridge_log = np.mean(test_ridge_log_folds, axis=0)
test_rtrans_log = np.mean(test_rtrans_log_folds, axis=0)

# =============================================================================
# Final predictions
# =============================================================================

test_blend_log = (
    w_cat * test_cat_log +
    w_lgbm * test_lgbm_log +
    w_ridge * test_ridge_log +
    w_rtrans * test_rtrans_log
)

test_blend = np.clip(np.expm1(test_blend_log), 0, None)
test_cat = np.clip(np.expm1(test_cat_log), 0, None)
test_lgbm = np.clip(np.expm1(test_lgbm_log), 0, None)
test_ridge = np.clip(np.expm1(test_ridge_log), 0, None)
test_rtrans = np.clip(np.expm1(test_rtrans_log), 0, None)

# =============================================================================
# Save submissions
# =============================================================================

pd.DataFrame({ID_COL: test_ids, TARGET: test_blend}).to_csv("submission_blend_v4.csv", index=False)
pd.DataFrame({ID_COL: test_ids, TARGET: test_cat}).to_csv("submission_catboost_v4.csv", index=False)
pd.DataFrame({ID_COL: test_ids, TARGET: test_lgbm}).to_csv("submission_lgbm_v4.csv", index=False)
pd.DataFrame({ID_COL: test_ids, TARGET: test_ridge}).to_csv("submission_ridge_meta_v4.csv", index=False)
pd.DataFrame({ID_COL: test_ids, TARGET: test_rtrans}).to_csv("submission_ridge_transcript_v4.csv", index=False)

print("\nAll submissions saved!")
print("  submission_blend_v4.csv        <- primary submission")
print("  submission_catboost_v4.csv")
print("  submission_lgbm_v4.csv")
print("  submission_ridge_meta_v4.csv")
print("  submission_ridge_transcript_v4.csv")


Fold 1/5
CatBoost        MSLE: 0.526664 (best_iter=612)
LightGBM        MSLE: 0.526973 (best_iter=556)
Ridge(meta)     MSLE: 1.061999
Ridge(trans)    MSLE: 1.088257

Fold 2/5
CatBoost        MSLE: 0.112880 (best_iter=652)
LightGBM        MSLE: 0.121089 (best_iter=432)
Ridge(meta)     MSLE: 0.547588
Ridge(trans)    MSLE: 0.581682

Fold 3/5
CatBoost        MSLE: 0.131723 (best_iter=1434)
LightGBM        MSLE: 0.143560 (best_iter=543)
Ridge(meta)     MSLE: 0.584360
Ridge(trans)    MSLE: 0.621786

Fold 4/5
CatBoost        MSLE: 0.143983 (best_iter=1100)
LightGBM        MSLE: 0.139837 (best_iter=262)
Ridge(meta)     MSLE: 0.600856
Ridge(trans)    MSLE: 0.653193

Fold 5/5
CatBoost        MSLE: 0.150071 (best_iter=359)
LightGBM        MSLE: 0.151057 (best_iter=157)
Ridge(meta)     MSLE: 0.440057
Ridge(trans)    MSLE: 0.515395

OOF MSLE Summary
CatBoost:         0.213064
LightGBM:         0.216503
Ridge(meta):      0.646972
Ridge(trans):     0.692062

Searching blend weights in log-space...

